# Spectral-geometric diffusion from scratch

This notebook implements the Euclidean diffusion tutorial from [Chenyang Yuan](https://chenyang.co/diffusion.html) and then extends it with the **ArrowSpace spectral-geometric metric** described in [blog post 021](https://www.tuned.org.uk/posts/021_diffusion_as_spectral_geometric_projection/).

## What we compare

1. **Baseline Euclidean diffusion**: $x_\\sigma = x_0 + \\sigma \\epsilon$, with $\\epsilon \\sim \\mathcal N(0, I)$.
2. **Spectral-geometric diffusion**: $x_\\sigma = x_0 + \\sigma M^{-1/2} \\epsilon$, where

$$M_{0.5} = \\tfrac12(I + \\Pi_F)$$

and $\\Pi_F$ is the low-frequency projector from the **feature-space Laplacian**.

The key correction from the original draft is that the **forward corruption must use the metric's covariance**.  If we only re-weight the reconstruction loss while keeping isotropic noise, the Bayes-optimal denoiser is still the Euclidean conditional mean.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, '../src')
from spectral_diffusion import (
    ScheduleLogLinear,
    Swissroll,
    TimeInputMLP,
    SpectralGeometry,
    build_spectral_geometry,
    IdealDenoiser,
    IdealSpectralDenoiser,
    training_loop_euclidean,
    training_loop_spectral,
    sample_euclidean,
    sample_spectral,
    samples_with_momentum,
    pairwise,
)

print('torch:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

## 1. Toy dataset: Swiss roll spiral

Same dataset used in the original tutorial.

In [ ]:
dataset = Swissroll(np.pi/2, 5*np.pi, 100)
loader = DataLoader(dataset, batch_size=2048, shuffle=True)

data_np = dataset.data.numpy()
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(data_np[:, 0], data_np[:, 1], s=8, alpha=0.7, c=np.linspace(0, 1, len(data_np)), cmap='viridis')
ax.set_aspect('equal')
ax.set_title('Swiss roll training data')
ax.set_xlabel('$x_0$'); ax.set_ylabel('$x_1$')
plt.tight_layout()
plt.show()

## 2. Baseline Euclidean diffusion

Train a small MLP exactly as in the tutorial.

In [ ]:
torch.manual_seed(3407)
model_euclidean = TimeInputMLP(dim=2, hidden_dims=(16, 128, 128, 128, 128, 16))
schedule = ScheduleLogLinear(N=200, sigma_min=0.005, sigma_max=10.0)

trainer = training_loop_euclidean(loader, model_euclidean, schedule, epochs=15000, device=device)
losses_euclidean = []
for i, ns in enumerate(trainer):
    losses_euclidean.append(ns['loss'])
    if i % 1000 == 0:
        print(f"step {i:6d}  loss {ns['loss']:.4f}")

# Plot smoothed loss
window = 200
loss_smooth = np.convolve(losses_euclidean, np.ones(window)/window, mode='valid')
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(loss_smooth)
ax.set_title('Euclidean training loss (moving average)')
ax.set_xlabel('gradient step'); ax.set_ylabel('MSE')
plt.tight_layout()
plt.show()

### 2a. Learned denoiser vector field

For a grid of noisy points, plot $\\hat x_0 = x - \\sigma \\epsilon_\\theta(x, \\sigma)$.

In [ ]:
model_euclidean.eval().to(device)

def plot_denoiser_field(model, sigmas_to_plot, title, geometry=None):
    fig, axes = plt.subplots(1, len(sigmas_to_plot), figsize=(4*len(sigmas_to_plot), 4))
    if len(sigmas_to_plot) == 1:
        axes = [axes]
    for ax, sig in zip(axes, sigmas_to_plot):
        x = torch.linspace(-12, 12, 20)
        y = torch.linspace(-12, 12, 20)
        xx, yy = torch.meshgrid(x, y, indexing='xy')
        grid = torch.stack([xx.flatten(), yy.flatten()], dim=1).to(device)
        sig_t = torch.full((grid.shape[0],), sig, device=device)
        with torch.no_grad():
            if geometry is None:
                eps = model(grid, sig_t)
                x0_hat = grid - sig * eps
            else:
                eps_M = model(grid, sig_t)
                eps = eps_M @ geometry.M_inv_sqrt.T
                x0_hat = grid - sig * eps
        x0_hat = x0_hat.cpu().numpy()
        grid_np = grid.cpu().numpy()
        ax.scatter(data_np[:, 0], data_np[:, 1], s=8, c='gray', alpha=0.3, label='data')
        ax.quiver(grid_np[:, 0], grid_np[:, 1],
                  x0_hat[:, 0] - grid_np[:, 0],
                  x0_hat[:, 1] - grid_np[:, 1],
                  scale=20, width=0.004, color='firebrick')
        ax.set_aspect('equal')
        ax.set_title(f'$\\sigma$ = {sig:.3f}')
        ax.set_xlim(-14, 14); ax.set_ylim(-14, 14)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

plot_denoiser_field(model_euclidean, [0.5, 2.0, 5.0], 'Euclidean learned denoiser field')

### 2b. Samples from Euclidean DDIM

In [ ]:
sigmas = schedule.sample_sigmas(20).to(device)
samples_euc = sample_euclidean(model_euclidean, sigmas, batchsize=2000, device=device).cpu().numpy()

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(samples_euc[:, 0], samples_euc[:, 1], s=5, alpha=0.4, c='coral')
ax.scatter(data_np[:, 0], data_np[:, 1], s=8, alpha=0.7, c='gray')
ax.set_aspect('equal')
ax.set_title('20-step DDIM samples (Euclidean)')
plt.tight_layout()
plt.show()

## 3. ArrowSpace spectral-geometric diffusion

Now build the feature-space Laplacian on the Swiss roll data.  Because the dataset is 2-D, the feature graph has only 2 nodes; we use this mainly to demonstrate the construction, but the same code works for high-dimensional item vectors.

In [ ]:
# Build spectral geometry from the full training set
X_all = torch.stack(list(dataset))
geometry = build_spectral_geometry(X_all, k=1, tau=0.5, r=None)
geometry = geometry.to(device)
print('Pi =\n', geometry.Pi)
print('M =\n', geometry.M)
print('M^{-1/2} =\n', geometry.M_inv_sqrt)

# Demonstrate the metric-matched corruption
x0_demo = X_all[:5]
sigma_demo = torch.ones(5)
x_sigma_demo, eps_demo = geometry.corrupt(x0_demo, sigma_demo)
print('\nIsotropic corruption std along feature 0:', (x0_demo + sigma_demo[:, None]*torch.randn_like(x0_demo) - x0_demo)[:, 0].std().item())
print('Metric-matched corruption std along feature 0:', (x_sigma_demo - x0_demo)[:, 0].std().item())

### 3a. Train the spectral-geometric model

In [ ]:
torch.manual_seed(3407)
model_spectral = TimeInputMLP(dim=2, hidden_dims=(16, 128, 128, 128, 128, 16))
schedule = ScheduleLogLinear(N=200, sigma_min=0.005, sigma_max=10.0)

trainer_spec = training_loop_spectral(
    loader, model_spectral, schedule, geometry,
    epochs=15000, device=device, loss_weight=0.5
)
losses_spectral = []
for i, ns in enumerate(trainer_spec):
    losses_spectral.append(ns['loss'])
    if i % 1000 == 0:
        print(f"step {i:6d}  loss {ns['loss']:.4f}  noise {ns['loss_noise']:.4f}  proj {ns['loss_projection']:.4f}")

window = 200
loss_smooth_spec = np.convolve(losses_spectral, np.ones(window)/window, mode='valid')
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(loss_smooth_spec, label='spectral')
ax.plot(loss_smooth[:len(loss_smooth_spec)], label='euclidean', alpha=0.7)
ax.set_title('Training loss comparison')
ax.set_xlabel('gradient step'); ax.set_ylabel('loss')
ax.legend()
plt.tight_layout()
plt.show()

### 3b. Spectral denoiser field

In [ ]:
plot_denoiser_field(model_spectral, [0.5, 2.0, 5.0], 'Spectral-geometric learned denoiser field', geometry=geometry)

### 3c. Samples from spectral-geometric DDIM

In [ ]:
sigmas = schedule.sample_sigmas(20).to(device)
samples_spec = sample_spectral(model_spectral, sigmas, geometry, batchsize=2000, device=device).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(samples_euc[:, 0], samples_euc[:, 1], s=5, alpha=0.4, c='coral')
axes[0].scatter(data_np[:, 0], data_np[:, 1], s=8, alpha=0.7, c='gray')
axes[0].set_aspect('equal'); axes[0].set_title('Euclidean DDIM')
axes[1].scatter(samples_spec[:, 0], samples_spec[:, 1], s=5, alpha=0.4, c='teal')
axes[1].scatter(data_np[:, 0], data_np[:, 1], s=8, alpha=0.7, c='gray')
axes[1].set_aspect('equal'); axes[1].set_title('Spectral-geometric DDIM')
for ax in axes:
    ax.set_xlim(-14, 14); ax.set_ylim(-14, 14)
fig.suptitle('Sample comparison')
plt.tight_layout()
plt.show()

## 4. Ideal denoisers and the role of metric-matched noise

For a small dataset we can compute the ideal denoiser in closed form.  Compare Euclidean vs spectral soft projections for the same noisy grid.

In [ ]:
ideal_euc = IdealDenoiser(dataset, device=device)
ideal_spec = IdealSpectralDenoiser(dataset, geometry, device=device)

sigmas_plot = [0.5, 2.0, 5.0]
fig, axes = plt.subplots(2, len(sigmas_plot), figsize=(4*len(sigmas_plot), 8))
for j, sig in enumerate(sigmas_plot):
    x = torch.linspace(-12, 12, 15)
    y = torch.linspace(-12, 12, 15)
    xx, yy = torch.meshgrid(x, y, indexing='xy')
    grid = torch.stack([xx.flatten(), yy.flatten()], dim=1).to(device)
    sig_t = torch.full((grid.shape[0],), sig, device=device)
    with torch.no_grad():
        eps_euc = ideal_euc(grid, sig_t)
        eps_spec = ideal_spec(grid, sig_t)
    grid_np = grid.cpu().numpy()
    # Euclidean row
    x0_hat_euc = (grid - sig * eps_euc).cpu().numpy()
    axes[0, j].scatter(data_np[:, 0], data_np[:, 1], s=8, c='gray', alpha=0.3)
    axes[0, j].quiver(grid_np[:, 0], grid_np[:, 1],
                      x0_hat_euc[:, 0] - grid_np[:, 0],
                      x0_hat_euc[:, 1] - grid_np[:, 1],
                      scale=20, width=0.004, color='firebrick')
    axes[0, j].set_aspect('equal')
    axes[0, j].set_title(f'$\\sigma$={sig:.2f}')
    # Spectral row
    x0_hat_spec = (grid - sig * eps_spec).cpu().numpy()
    axes[1, j].scatter(data_np[:, 0], data_np[:, 1], s=8, c='gray', alpha=0.3)
    axes[1, j].quiver(grid_np[:, 0], grid_np[:, 1],
                      x0_hat_spec[:, 0] - grid_np[:, 0],
                      x0_hat_spec[:, 1] - grid_np[:, 1],
                      scale=20, width=0.004, color='teal')
    axes[1, j].set_aspect('equal')
axes[0, 0].set_ylabel('Euclidean ideal denoiser')
axes[1, 0].set_ylabel('Spectral ideal denoiser')
fig.suptitle('Ideal denoiser fields')
plt.tight_layout()
plt.show()

## 5. Why metric-matched noise matters

Train two models with the **same spectral loss** but different corruption:
- `model_iso`: isotropic noise + spectral reconstruction loss.
- `model_metric`: metric-matched noise + spectral reconstruction loss.

If the theory is correct, only `model_metric` will learn a spectral-geometric score.

In [ ]:
def training_loop_spectral_isotropic(loader, model, schedule, geometry, epochs, device):
    # Spectral loss but isotropic forward corruption.
    model = model.to(device)
    geometry = geometry.to(device)
    optimizer = torch.optim.Adam(model.parameters())
    for epoch in range(epochs):
        for x0 in loader:
            x0 = x0.to(device)
            optimizer.zero_grad()
            sigma = schedule.sample_batch(x0)
            eps = torch.randn_like(x0)
            x_sigma = x0 + sigma[:, None] * eps
            eps_hat = model(x_sigma, sigma)
            x_hat = x_sigma - sigma[:, None] * eps_hat
            loss = geometry.reconstruction_loss(x_hat, x0)
            loss.backward()
            optimizer.step()
            yield {'loss': loss.item()}

torch.manual_seed(3407)
model_iso = TimeInputMLP(dim=2, hidden_dims=(16, 128, 128, 128, 128, 16))
trainer_iso = training_loop_spectral_isotropic(loader, model_iso, schedule, geometry, epochs=15000, device=device)
losses_iso = []
for i, ns in enumerate(trainer_iso):
    losses_iso.append(ns['loss'])
    if i % 1000 == 0:
        print(f"step {i:6d}  loss {ns['loss']:.4f}")

print(f"final isotropic+loss loss: {losses_iso[-1]:.4f}")

# Quick sample from isotropic-loss model (interpret it as Euclidean DDIM)
sigmas = schedule.sample_sigmas(20).to(device)
samples_iso = sample_euclidean(model_iso, sigmas, batchsize=2000, device=device).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(samples_iso[:, 0], samples_iso[:, 1], s=5, alpha=0.4, c='purple')
axes[0].scatter(data_np[:, 0], data_np[:, 1], s=8, alpha=0.7, c='gray')
axes[0].set_aspect('equal'); axes[0].set_title('Isotropic noise + spectral loss')
axes[1].scatter(samples_spec[:, 0], samples_spec[:, 1], s=5, alpha=0.4, c='teal')
axes[1].scatter(data_np[:, 0], data_np[:, 1], s=8, alpha=0.7, c='gray')
axes[1].set_aspect('equal'); axes[1].set_title('Metric-matched noise + spectral loss')
for ax in axes:
    ax.set_xlim(-14, 14); ax.set_ylim(-14, 14)
fig.suptitle('The forward corruption must match the metric')
plt.tight_layout()
plt.show()

## 6. Momentum sampler comparison

Use the generalised sampler from the tutorial with $\gamma=2, \mu=0$ (gradient estimation) on both models.

In [ ]:
sigmas_long = schedule.sample_sigmas(20).to(device)

@torch.no_grad()
def sample_with_model(model, sigmas, batchsize=2000, gam=2.0, mu=0.0):
    model = model.to(device)
    xs = list(samples_with_momentum(model, sigmas, gam=gam, mu=mu, batchsize=batchsize, device=device))
    return xs[-1].cpu().numpy()

samples_euc_mom = sample_with_model(model_euclidean, sigmas_long, gam=2.0, mu=0.0)
samples_spec_mom = sample_with_model(model_spectral, sigmas_long, gam=2.0, mu=0.0)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(samples_euc_mom[:, 0], samples_euc_mom[:, 1], s=5, alpha=0.4, c='coral')
axes[0].scatter(data_np[:, 0], data_np[:, 1], s=8, alpha=0.7, c='gray')
axes[0].set_aspect('equal'); axes[0].set_title('Euclidean + momentum')
axes[1].scatter(samples_spec_mom[:, 0], samples_spec_mom[:, 1], s=5, alpha=0.4, c='teal')
axes[1].scatter(data_np[:, 0], data_np[:, 1], s=8, alpha=0.7, c='gray')
axes[1].set_aspect('equal'); axes[1].set_title('Spectral + momentum')
for ax in axes:
    ax.set_xlim(-14, 14); ax.set_ylim(-14, 14)
fig.suptitle('Gradient-estimation sampler ($\\gamma=2$, $\\mu=0$)')
plt.tight_layout()
plt.show()

## Takeaways

1. **The metric must enter the forward process.**  Isotropic noise + spectral loss still learns an effectively Euclidean score; metric-matched noise is required for the denoiser to encode the ArrowSpace feature-manifold geometry.

2. **The spectral-geometric sampler** uses $M^{-1/2}$ both at initialization and when converting the model's prediction $M^{1/2}\\epsilon$ back to raw-coordinate updates.

3. **Same API, richer geometry.**  The training loop and sampler stay almost identical to the Euclidean tutorial; only the corruption and the initial noise distribution change.
